# No shock general coupling 
Calculate across groups (general) coupling for no shock model


In [5]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
from utils import plot_pd          # 
# add random seed for reproducibility
random_seed = 42


In [6]:

# --- load and prep data (mirrors analysis_script.py) ---
df = pd.read_csv("data/scr_amg_hipp_all_noSHock.csv")

df['sub_idx'] = pd.Categorical(df['sub']).codes
n_subs = df['sub_idx'].nunique()

sub_idx  = df['sub_idx'].values
amg      = df['amg'].values
trialNo  = df['trialNo'].values
pe       = df['pe'].values


def fit_pooled(coupling_col):
    coupling = df[coupling_col].values
    with pm.Model() as m:
        beta_coupling = pm.Normal('beta_coupling', 0, 1)
        beta_amg      = pm.Normal('beta_amg', 0, 1)
        beta_trialNo  = pm.Normal('beta_trialNo', 0, 1)
        mu_a    = pm.Normal('mu_a', 0, 1)
        sigma_a = pm.HalfNormal('sigma_a', 1)
        z_a = pm.Normal('z_a', 0, 1, shape=n_subs)
        a   = pm.Deterministic('a', mu_a + z_a * sigma_a)
        mu = (a[sub_idx]
              + beta_coupling * coupling
              + beta_amg * amg
              + beta_trialNo * trialNo)
        sigma = pm.HalfNormal('sigma', 1)
        pm.Normal('pe', mu=mu, sigma=sigma, observed=pe)
        tr = pm.sample(chains=4, return_inferencedata=True,
                       idata_kwargs={"log_likelihood": True}, random_seed=random_seed)
    return tr


trace_hipp_pooled  = fit_pooled('coupling')
trace_vmpfc_pooled = fit_pooled('amg_vmpfc')

slope_hipp_pooled  = trace_hipp_pooled.posterior['beta_coupling']
slope_vmpfc_pooled = trace_vmpfc_pooled.posterior['beta_coupling']


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_coupling, beta_amg, beta_trialNo, mu_a, sigma_a, z_a, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 5 seconds.
There were 5 divergences after tuning. Increase `target_accept` or reparameterize.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_coupling, beta_amg, beta_trialNo, mu_a, sigma_a, z_a, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 5 seconds.


In [7]:

# sanity check against the manuscript (β≈0.10 / −0.03, pd 100% / 95.5%)
for name, s in [("hipp", slope_hipp_pooled), ("vmpfc", slope_vmpfc_pooled)]:
    v = s.values.ravel()
    print(name, "mean=%.3f" % v.mean(),
          "sd=%.3f" % v.std(),
        "pd=%.1f%%" % (max((v>0).mean(), (v<0).mean())*100),
        "89%HDI=", az.hdi(v, hdi_prob=0.89))



hipp mean=0.102 sd=0.019 pd=100.0% 89%HDI= [0.07403113 0.13250339]
vmpfc mean=-0.043 sd=0.016 pd=99.8% 89%HDI= [-0.06890116 -0.01894878]
